# L1 - OpenAI function calling

In [2]:
import sys
import os
import openai

# Use current working directory and go one level up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

# Now you can import your config
from config import api_key

#openai.api_key = api_key
os.environ['OPENAI_API_KEY'] = api_key

#### Step 1 - create a python function that can call for example a real api

In [3]:
import json

# Example dummy function hard coded to return the same weather
# In production, this could be your backend API or an external API
def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location"""
    weather_info = {
        "location": location,
        "temperature": "72",
        "unit": unit,
        "forecast": ["sunny", "windy"],
    }
    return json.dumps(weather_info)

#### Step 2 - Create inputs to funciton under Step 1

Create a function that let an LLM create the inputs for the function under step 1

In [ ]:
# define a function
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location"],
        },
    }
]

#### Step 3 - create prompt and invoke llm

In [ ]:
messages = [
    {
        "role": "user",
        "content": "What's the weather like in Boston?"
    }
]

In [ ]:
from openai import OpenAI

client = OpenAI()

# Call the ChatCompletion endpoint
response = openai.chat.completions.create(
    # OpenAI Updates: As of June 2024, we are now using the GPT-3.5-Turbo model
    model="gpt-3.5-turbo",
    messages=messages,
    functions=functions
)

In [ ]:
print(response)

> Note: The following result may differ slightly from the one shown by the instructor in the video lesson due to the model being updated.

In [ ]:
response.choices[0].message.content

In [ ]:
response.choices[0].message.function_call

In [ ]:
args = response.choices[0].message.function_call.arguments

#### Step 4 - call the function (Step1) that perfoms the action

In [ ]:
get_current_weather(args)

* Pass a message that is not related to a function.

In [ ]:
messages = [
    {
        "role": "user",
        "content": "hi!",
    }
]

In [ ]:
# Call the ChatCompletion endpoint
response = openai.chat.completions.create(
    # OpenAI Updates: As of June 2024, we are now using the GPT-3.5-Turbo model
    model="gpt-3.5-turbo",
    messages=messages,
    functions=functions
)

In [ ]:
import pprint
pprint.pprint(response.model_dump())

In [ ]:
response.choices[0].message.content

### Settings to determine which function to use

* Pass additional parameters to force the model to use or not a function.
  > function_call = "none", "auto", {"name": "<function name string>"}
* Play with the argumnents and check the output

In [ ]:
messages = [
    {
        "role": "user",
        "content": "hi",
    }
]
response = client.chat.completions.create(
    # OpenAI Updates: As of June 2024, we are now using the GPT-3.5-Turbo model
    model="gpt-3.5-turbo",
    messages=messages,
    functions=functions,
    # this argument can be used to use to determine function call
    # default is auto
    function_call={"name": "get_current_weather"},
)
pprint.pprint(response.model_dump())

# LangChain Expression Language (LCEL)

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.schema.output_parser import StrOutputParser

In [ ]:
prompt = ChatPromptTemplate.from_template(
    "tell me a short joke about {topic}"
)

# define the langchain openai 
model = ChatOpenAI(temperature=1.0)

# define the output parser
output_parser = StrOutputParser()

In [ ]:
chain_no_parser = prompt | model

In [ ]:
response = chain_no_parser.invoke({"topic": "bears"})
response.content

In [ ]:
chain = prompt | model | output_parser
chain.invoke({"topic": "bears"})

## Embeddings 

In [ ]:
from langchain.vectorstores import DocArrayInMemorySearch
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [ ]:
model = ChatOpenAI()

vectorstore = DocArrayInMemorySearch.from_texts(
    [
        "harrisson worked at kensho", 
         "bears like to eat honey"
    ],
    embedding = OpenAIEmbeddings())

retriever = vectorstore.as_retriever()

In [ ]:
# This will retreive the most similar article stored in the
# in memory verctor database
retriever.invoke("where did harrison work?")

In [ ]:
retriever.invoke("what do bears like to eat?")

In [ ]:
template = """Anser the question only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [ ]:
from langchain.schema.runnable import RunnableMap
from langchain.schema.output_parser import StrOutputParser

output_parser = StrOutputParser()

Step 1. The question is converted to an embedding and then the best matches are obtained from the retreiver. This will be used as context.

Step 2. The context is used by the LLM together with the question to generate the final answer.

In [ ]:
chain = RunnableMap({
    "context": lambda x: retriever.invoke(x['question']),
    "question" : lambda x: x["question"]}) | prompt | model | output_parser

chain.invoke({"question" : "where did harrison work?"})

You can also only run the `RunnableMap` to see its output

In [ ]:
inputs = RunnableMap({
    "context": lambda x: retriever.invoke(x['question']),
    "question" : lambda x: x["question"]}) 

inputs.invoke({"question" : "where did harrison work?"})

## Fallbacks

My understanding is that the fallback a backup in case there are error messages.

In [ ]:
from langchain_openai import OpenAI
import json

create a simple model

In [ ]:
simple_model = OpenAI(
    temperature=0, 
    max_tokens=1000, 
    model="gpt-3.5-turbo-instruct"
)

create a simple chain

In [ ]:
simple_chain = simple_model | json.loads

first run the prompt with the simple model. This should always work

In [ ]:
challenge = "write three poems in a json blob, where each poem is a json blob of a title, author, and first line"
#challenge = "write three poems in a json blob, where each poem is a json blob of a title, author, and first line. Return only the string that can be loaded via json.loads"
response = simple_model.invoke(challenge)

The returned by newer models is a json string. However this is not guarenteed with LLM's.

In [ ]:
# This is not valid json
print(response)

In [ ]:
# fails because the chain expects a valid json object
#simple_chain.invoke(challenge)

<p style=\"background-color:#F5C780; padding:15px\"><b>Note:</b> The next line is expected to fail.</p>

In [ ]:
model = ChatOpenAI(temperature=0, model='gpt-3.5-turbo')
chain = model | StrOutputParser() | json.loads

In [ ]:
chain.invoke(challenge)

Here we run `simple_chain` and if it fails it will use `chain` as backup

In [ ]:
final_chain = simple_chain.with_fallbacks([chain])
final_chain.invoke(challenge)

## Interface

Several ways to invoke prompts

In [ ]:
prompt = ChatPromptTemplate.from_template(
    "Tell me a short joke about {topic}"
)
model = ChatOpenAI()
output_parser = StrOutputParser()

chain = prompt | model | output_parser

single prompt call

In [ ]:
chain.invoke({"topic": "bears"})

multi prompt call

In [ ]:
chain.batch([{"topic": "bears"}, {"topic": "frogs"}])

In [ ]:
for t in chain.stream({"topic": "bears"}):
    print(t)

In [ ]:
response = await chain.ainvoke({"topic": "bears"})
response

# OpenAI functions using LangChain

In [ ]:
from typing import List
from pydantic import BaseModel, Field

## Pydantic Syntax

Pydantic data classes are a blend of Python's data classes with the validation power of Pydantic. 

They offer a concise way to define data structures while ensuring that the data adheres to specified types and constraints.

In standard python you would create a **class** like this:

In [ ]:
class User:
    def __init__(self, name: str, age: int, email: str):
        self.name = name
        self.age = age
        self.email = email

you can create attributes that do not match the indicated datatype

In [ ]:
foo = User(name="Joe",age="32", email="joe@gmail.com")

In [ ]:
foo.name

This will not result in an error during run time. Only packages that check this will flag this.

#### Use pydantic to enforce datatypes

In [ ]:
class pUser(BaseModel):
    name: str
    age: int
    email: str

In [ ]:
foo_p = pUser(name="Jane", age=32, email="jane@gmail.com")

In [ ]:
foo_p = pUser(name="Jane", age="32", email="jane@gmail.com")

pydantic can also partially broadcast to the correct value. Here it converts the string to an integer

In [ ]:
type(foo_p.age)

This will fail and result in a validation Error

In [ ]:
# foo_p = pUser(name="Jane", age="bar", email="jane@gmail.com")

## Pydantic to OpenAI function definition

In [ ]:
class WeatherSearch(BaseModel):
    """Call this with an airport code to get the weather at that airport"""
    airport_code: str = Field(...,description="airport code to get weather for")

This defines a class `WeatherSearch` that inherets from the pydantic BaseModel.

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_function
import pprint

LangChain can be used to create a openai function from the `WeatherSearch` class

In [ ]:
weather_function = convert_to_openai_function(WeatherSearch)
pprint.pprint(weather_function)

### Use the function using lanchain

In [ ]:
from langchain_openai import ChatOpenAI

#### Method 1

In [ ]:
model = ChatOpenAI()
model.invoke("what is the weather in SF today?", functions=[weather_function])

#### Method 2

In [ ]:
model_with_functions = model.bind(functions=[weather_function])
model_with_functions.invoke("what is the weather in sf?")

#### with forced binding

The argument `function_call` is the same as in the chapter openai function calling.

In [ ]:
model_with_forced_function = model.bind(functions=[weather_function], function_call={"name":"WeatherSearch"})
model_with_forced_function.invoke("what is the weather in sf?")

This will force the use of the function and it will hallucenate inputs in case these are not available in the prompt

In [ ]:
model_with_forced_function.invoke("hi!")

## Using the chain

In [ ]:
from langchain.prompts import ChatPromptTemplate

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "{input}")
])

In [ ]:
prompt.messages

In [ ]:
chain = prompt | model_with_functions
chain.invoke({"input":"what is the weather in sf"})

## Multiple functions

In [ ]:
class WeatherSearch(BaseModel):
    """Call this with an airport code to get the weather at that airport"""
    airport_code: str = Field(...,description="airport code to get weather for")

class ArtistSearch(BaseModel):
    """Call this to get the names of songs by a particular artist"""
    artist_name: str = Field(description="name of artist to look up")
    n: int = Field(description="number of results")

In [ ]:
# create a list of functions
functions = [
    convert_to_openai_function(WeatherSearch),
    convert_to_openai_function(ArtistSearch),
]

# bind the functions to the model
model_with_functions = model.bind(functions=functions)

# invoke the model
model_with_functions.invoke("what is the weather in sf?").additional_kwargs

In [ ]:
model_with_functions.invoke("what are three songs by taylor swift?").additional_kwargs

In [ ]:
model_with_functions.invoke("hi!").content

# Tagging and Extraction Using OpenAI functions

> **Note** - Please note that tagging and extraction are not really running api's that connect to the outside world. The tagging and extraction are operations that LLM's perform directly on the text that is provided.

Tagging is more about adding certain labels to the data. For example on can determine the sentiment and language of a piece of text.

In [ ]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.utils.function_calling import convert_to_openai_function

In [ ]:
class Tagging(BaseModel):
    """Tag the piece of text with particular info."""
    sentiment: str = Field(description="sentiment of text, should be `pos`, `neg`, or `neutral`")
    language: str = Field(description="language of text (should be ISO 639-1 code)")

In [ ]:
convert_to_openai_function(Tagging)

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [ ]:
model = ChatOpenAI(temperature=0)

In [ ]:
tagging_functions = [convert_pydantic_to_openai_function(Tagging)]

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Think carefully, and then tag the text as instructed"),
    ("user", "{input}")
])

In [ ]:
from langchain_core.output_parsers import StrOutputParser

model_with_functions = model.bind(
    functions=tagging_functions,
    function_call={"name": "Tagging"}
)

output_parser = StrOutputParser()

tagging_chain = prompt | model_with_functions 

In [ ]:
response = tagging_chain.invoke({"input": "Ik vind LangChain erg goed."})
response.additional_kwargs['function_call']

In [ ]:
response = tagging_chain.invoke({"input": "non mi piace questo cibo"})
response.additional_kwargs['function_call']

In [ ]:
from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser

In [ ]:
tagging_chain = prompt | model_with_functions | JsonOutputFunctionsParser()
tagging_chain.invoke({"input": "non mi piace questo cibo"})

## Extraction

Extraction is similar to tagging, but used for extracting multiple pieces of information.

In [ ]:
from typing import Optional
class Person(BaseModel):
    """Information about a person."""
    name: str = Field(description="person's name")
    age: Optional[int] = Field(description="person's age")

In [ ]:
class Information(BaseModel):
    """Information to extract."""
    people: List[Person] = Field(description="List of info about people")

In [ ]:
convert_pydantic_to_openai_function(Information)

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the relevant information, if not explicitly provided do not guess. Extract partial info"),
    ("human", "{input}")])


extraction_functions = [convert_pydantic_to_openai_function(Information)]
extraction_model = model.bind(functions=extraction_functions, function_call={"name": "Information"})

response = extraction_model.invoke("Joe is 30, his mom is Martha")
response.additional_kwargs['function_call']

In [ ]:
response.additional_kwargs['function_call']

> parse the output directly

In [ ]:
extraction_chain = prompt | extraction_model | JsonOutputFunctionsParser()
extraction_chain.invoke({"input": "Joe is 30, his mom is Martha"})

> parse the output to remove the `people` key.

In [ ]:
from langchain.output_parsers.openai_functions import JsonKeyOutputFunctionsParser
extraction_chain = prompt | extraction_model | JsonKeyOutputFunctionsParser(key_name="people")
extraction_chain.invoke({"input": "Joe is 30, his mom is Martha"})

## Doing it for real

We can apply tagging to a larger body of text.

For example, let's load this blog post and extract tag information from a sub-set of the text.

In [ ]:
from langchain.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
documents = loader.load()

In [ ]:
doc = documents[0]

In [ ]:
page_content = doc.page_content[:10000]
# print(page_content[:1000].strip())

In [ ]:
class Overview(BaseModel):
    """Overview of a section of text."""
    summary: str = Field(description="Provide a concise summary of the content.")
    language: str = Field(description="Provide the language that the content is written in.")
    keywords: str = Field(description="Provide keywords related to the content.")

In [ ]:
overview_tagging_function = [
    convert_pydantic_to_openai_function(Overview)
]
tagging_model = model.bind(
    functions=overview_tagging_function,
    function_call={"name":"Overview"}
)
tagging_chain = prompt | tagging_model | JsonOutputFunctionsParser()

tagging_chain.invoke({"input": page_content})

In [ ]:
class Paper(BaseModel):
    """Information about papers mentioned."""
    title: str
    author: Optional[str]

class Info(BaseModel):
    """Information to extract"""
    papers: List[Paper]

In [ ]:
paper_extraction_function = [
    convert_pydantic_to_openai_function(Info)
]
extraction_model = model.bind(
    functions=paper_extraction_function, 
    function_call={"name":"Info"}
)

extraction_chain = prompt | extraction_model | JsonKeyOutputFunctionsParser(key_name="papers")

extraction_chain.invoke({"input": page_content})

In [ ]:
template = """A article will be passed to you. Extract from it all papers that are mentioned by this article follow by its author. 

Do not extract the name of the article itself. If no papers are mentioned that's fine - you don't need to extract any! Just return an empty list.

Do NOT make up or guess ANY extra information. Only extract what exactly is in the text."""

prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("human", "{input}")
])

In [ ]:
extraction_chain = prompt | extraction_model | JsonKeyOutputFunctionsParser(key_name="papers")
extraction_chain.invoke({"input": page_content})

In [ ]:
extraction_chain.invoke({"input": "hi"})

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_overlap=0)

# Tools and Routing

In [ ]:
from langchain.agents import tool

In [ ]:
@tool
def search(query: str) -> str:
    """Search for weather online"""
    return "42f"

In [ ]:
search.name

In [ ]:
search.description

In [ ]:
search.args

In [ ]:
from pydantic import BaseModel, Field
class SearchInput(BaseModel):
    query: str = Field(description="Thing to search for")


In [ ]:
@tool(args_schema=SearchInput)
def search(query: str) -> str:
    """Search for the weather online."""
    return "42f"

In [ ]:
search.args

In [ ]:
search.run("sf")

In [ ]:
import requests
from pydantic import BaseModel, Field
import datetime

# Define the input schema
class OpenMeteoInput(BaseModel):
    latitude: float = Field(..., description="Latitude of the location to fetch weather data for")
    longitude: float = Field(..., description="Longitude of the location to fetch weather data for")

@tool(args_schema=OpenMeteoInput)
def get_current_temperature(latitude: float, longitude: float) -> dict:
    """Fetch current temperature for given coordinates."""
    
    BASE_URL = "https://api.open-meteo.com/v1/forecast"
    
    # Parameters for the request
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'hourly': 'temperature_2m',
        'forecast_days': 1,
    }

    # Make the request
    response = requests.get(BASE_URL, params=params)
    
    if response.status_code == 200:
        results = response.json()
    else:
        raise Exception(f"API Request failed with status code: {response.status_code}")

    current_utc_time = datetime.datetime.utcnow()
    time_list = [datetime.datetime.fromisoformat(time_str.replace('Z', '+00:00')) for time_str in results['hourly']['time']]
    temperature_list = results['hourly']['temperature_2m']
    
    closest_time_index = min(range(len(time_list)), key=lambda i: abs(time_list[i] - current_utc_time))
    current_temperature = temperature_list[closest_time_index]
    
    return f'The current temperature is {current_temperature}°C'

In [ ]:
get_current_temperature.name

In [ ]:
get_current_temperature.description

In [ ]:
get_current_temperature.args

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_function

In [ ]:
convert_to_openai_function(get_current_temperature)

In [ ]:
get_current_temperature.invoke({"latitude": 13, "longitude": 14})

In [ ]:
import wikipedia
@tool
def search_wikipedia(query: str) -> str:
    """Run Wikipedia search and get page summaries."""
    page_titles = wikipedia.search(query)
    summaries = []
    for page_title in page_titles[: 3]:
        try:
            wiki_page =  wikipedia.page(title=page_title, auto_suggest=False)
            summaries.append(f"Page: {page_title}\nSummary: {wiki_page.summary}")
        except (
            self.wiki_client.exceptions.PageError,
            self.wiki_client.exceptions.DisambiguationError,
        ):
            pass
    if not summaries:
        return "No good Wikipedia Search Result was found"
    return "\n\n".join(summaries)

In [ ]:
search_wikipedia.name

In [ ]:
search_wikipedia.description

In [ ]:
convert_to_openai_function(search_wikipedia)

In [ ]:
print(search_wikipedia.invoke({"query": "langchain"}))

### Routing

In lesson 3, we show m of function calling deciding between two candidate functions.

Given our tools above, let's format these as OpenAI functions and show this same behavior.

In [ ]:
from langchain_openai import ChatOpenAI

functions = [
    format_tool_to_openai_function(f) for f in [
        search_wikipedia, get_current_temperature
    ]
]

model = ChatOpenAI(temperature=0).bind(functions=functions)

In [ ]:
# functions

In [ ]:
model.invoke("what is the weather in sf right now").additional_kwargs

In [ ]:
model.invoke("what is langchain").additional_kwargs

In [ ]:
from langchain.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful but sassy assistant"),
    ("user", "{input}"),
])
chain = prompt | model

In [ ]:
chain.invoke({"input": "what is the weather in sf right now"}).additional_kwargs

In [ ]:
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser

In [ ]:
chain = prompt | model | OpenAIFunctionsAgentOutputParser()

In [ ]:
result = chain.invoke({"input": "what is the weather in sf right now"})
result

In [ ]:
type(result)

In [ ]:
result.tool

In [ ]:
result.tool_input

In [ ]:
get_current_temperature(result.tool_input)

In [ ]:
result = chain.invoke({"input": "hi!"})

In [ ]:
type(result)

In [ ]:
result.return_values

> **Note**: below is important!!! Here you link the output of the function call to the real python funciton to execute it.

In [ ]:
from langchain.schema.agent import AgentFinish

def route(result):
    if isinstance(result, AgentFinish):
        return result.return_values['output']
    else:
        tools = {
            "search_wikipedia": search_wikipedia, 
            "get_current_temperature": get_current_temperature,
        }
        return tools[result.tool].run(result.tool_input)

In [ ]:
chain = prompt | model | OpenAIFunctionsAgentOutputParser()
result = chain.invoke({"input": "What is the weather in san francisco right now?"})
result

Here the output is basically the arguments that we need as input to the function that actually requires this inputs. The functions however is **not** executed.

> **Note** - below the `route` function is added to the chain. This means that code in this function is exectured based on the output of the language model

In [ ]:
chain = prompt | model | OpenAIFunctionsAgentOutputParser() | route
result = chain.invoke({"input": "What is the weather in san francisco right now?"})
result

In the next example the output of the llm is input to the wikipedia function

In [ ]:
result = chain.invoke({"input": "What is langchain?"})
result

In [ ]:
chain.invoke({"input": "hi!"})

# L6 - Conversational agent

In [4]:
from langchain.tools import tool

In [5]:
import requests
from pydantic import BaseModel, Field
import datetime

# Define the input schema
class OpenMeteoInput(BaseModel):
    latitude: float = Field(..., description="Latitude of the location to fetch weather data for")
    longitude: float = Field(..., description="Longitude of the location to fetch weather data for")

@tool(args_schema=OpenMeteoInput)
def get_current_temperature(latitude: float, longitude: float) -> dict:
    """Fetch current temperature for given coordinates."""
    
    BASE_URL = "https://api.open-meteo.com/v1/forecast"
    
    # Parameters for the request
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'hourly': 'temperature_2m',
        'forecast_days': 1,
    }

    # Make the request
    response = requests.get(BASE_URL, params=params)
    
    if response.status_code == 200:
        results = response.json()
    else:
        raise Exception(f"API Request failed with status code: {response.status_code}")

    current_utc_time = datetime.datetime.utcnow()
    time_list = [datetime.datetime.fromisoformat(time_str.replace('Z', '+00:00')) for time_str in results['hourly']['time']]
    temperature_list = results['hourly']['temperature_2m']
    
    closest_time_index = min(range(len(time_list)), key=lambda i: abs(time_list[i] - current_utc_time))
    current_temperature = temperature_list[closest_time_index]
    
    return f'The current temperature is {current_temperature}°C'

In [6]:
import wikipedia

@tool
def search_wikipedia(query: str) -> str:
    """Run Wikipedia search and get page summaries."""
    page_titles = wikipedia.search(query)
    summaries = []
    for page_title in page_titles[: 3]:
        try:
            wiki_page =  wikipedia.page(title=page_title, auto_suggest=False)
            summaries.append(f"Page: {page_title}\nSummary: {wiki_page.summary}")
        except (
            self.wiki_client.exceptions.PageError,
            self.wiki_client.exceptions.DisambiguationError,
        ):
            pass
    if not summaries:
        return "No good Wikipedia Search Result was found"
    return "\n\n".join(summaries)

In [7]:
tools = [get_current_temperature, search_wikipedia]

In [10]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser
from langchain_core.utils.function_calling import convert_to_openai_function

In [73]:
# Define the function list
functions = [convert_to_openai_function(f) for f in tools]

# Create model and bind funtions
model = ChatOpenAI(temperature=0).bind(functions=functions)

# Create prompts
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful but sassy assistant"),
    ("user", "{input}"),
])

# define chain
chain = prompt | model | OpenAIFunctionsAgentOutputParser()

In [74]:
result = chain.invoke({"input": "what is the weather is sf?"})

In [75]:
result.tool

'get_current_temperature'

In [76]:
result.tool_input

{'latitude': 37.7749, 'longitude': -122.4194}

In [77]:
result

AgentActionMessageLog(tool='get_current_temperature', tool_input={'latitude': 37.7749, 'longitude': -122.4194}, log="\nInvoking: `get_current_temperature` with `{'latitude': 37.7749, 'longitude': -122.4194}`\n\n\n", message_log=[AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"latitude":37.7749,"longitude":-122.4194}', 'name': 'get_current_temperature'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 112, 'total_tokens': 137, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'function_call', 'logprobs': None}, id='run--fc00c079-0792-4b5b-84a6-c44ca0d81eba-0', usage_metadata={'input_tokens': 112, 'output_tokens': 25, 'total_tokens': 137, 'input_token_details': {'audio': 0, 'cache_r

Check what `MessagesPlaceholder`

In [78]:
from langchain.prompts import MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful but sassy assistant"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [79]:
chain = prompt | model | OpenAIFunctionsAgentOutputParser()

In [80]:
result1 = chain.invoke({
    "input": "what is the weather is sf?",
    "agent_scratchpad": []
})

In [81]:
result1.tool

'get_current_temperature'

In [82]:
result1.tool_input

{'latitude': 37.7749, 'longitude': -122.4194}

In [41]:
observation = get_current_temperature.invoke(result1.tool_input)
observation

'The current temperature is 20.3°C'

In [42]:
type(result1)

langchain_core.agents.AgentActionMessageLog

In [43]:
from langchain.agents.format_scratchpad import format_to_openai_functions

In [44]:
result1.message_log[0].additional_kwargs

{'function_call': {'arguments': '{"latitude":37.7749,"longitude":-122.4194}',
  'name': 'get_current_temperature'},
 'refusal': None}

This combines the result1 and observation in two messages.

In [48]:
format_to_openai_functions([(result1, observation), ])[0]

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"latitude":37.7749,"longitude":-122.4194}', 'name': 'get_current_temperature'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 112, 'total_tokens': 137, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'function_call', 'logprobs': None}, id='run--0a926352-e5c6-4df7-a102-b5444c98af8a-0', usage_metadata={'input_tokens': 112, 'output_tokens': 25, 'total_tokens': 137, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [49]:
format_to_openai_functions([(result1, observation), ])[1]

FunctionMessage(content='The current temperature is 20.3°C', additional_kwargs={}, response_metadata={}, name='get_current_temperature')

In [51]:
result2 = chain.invoke({
    "input": "what is the weather is sf?", 
    "agent_scratchpad": format_to_openai_functions([(result1, observation)])
})

combine the inputs to get an agent finish result

In [52]:
result2

AgentFinish(return_values={'output': 'The current temperature in San Francisco is 20.3°C.'}, log='The current temperature in San Francisco is 20.3°C.')

In [53]:
from langchain.schema.agent import AgentFinish
def run_agent(user_input):
    intermediate_steps = []
    while True:
        result = chain.invoke({
            "input": user_input, 
            "agent_scratchpad": format_to_openai_functions(intermediate_steps)
        })
        if isinstance(result, AgentFinish):
            return result
        tool = {
            "search_wikipedia": search_wikipedia, 
            "get_current_temperature": get_current_temperature,
        }[result.tool]
        observation = tool.run(result.tool_input)
        intermediate_steps.append((result, observation))

> **Note** - the below is quite key. Maybe revisit it later on again.

In [66]:
from langchain.schema.runnable import RunnablePassthrough

# Define the function list
functions = [convert_to_openai_function(f) for f in tools]

# Create model and bind funtions
model = ChatOpenAI(temperature=0).bind(functions=functions)

chain = prompt | model | OpenAIFunctionsAgentOutputParser()

agent_chain = RunnablePassthrough.assign(
    agent_scratchpad= lambda x: format_to_openai_functions(x["intermediate_steps"])
) | chain

In [62]:
def run_agent(user_input):
    intermediate_steps = []
    while True:
        result = agent_chain.invoke({
            "input": user_input, 
            "intermediate_steps": intermediate_steps
        })
        if isinstance(result, AgentFinish):
            return result
        tool = {
            "search_wikipedia": search_wikipedia, 
            "get_current_temperature": get_current_temperature,
        }[result.tool]
        observation = tool.run(result.tool_input)
        intermediate_steps.append((result, observation))

In [56]:
run_agent("what is the weather is sf?")

AgentFinish(return_values={'output': 'The current temperature in San Francisco is 21.5°C.'}, log='The current temperature in San Francisco is 21.5°C.')

In [67]:
run_agent("What is the capital of France and what’s the temperature there?")

AgentFinish(return_values={'output': 'The current temperature in Paris, France is 21.0°C.'}, log='The current temperature in Paris, France is 21.0°C.')

In [68]:
run_agent("what is langchain?")

AgentFinish(return_values={'output': 'LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. It is used for document analysis and summarization, chatbots, and code analysis.'}, log='LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. It is used for document analysis and summarization, chatbots, and code analysis.')

In [58]:
run_agent("hi!")

AgentFinish(return_values={'output': 'Well, hello there! How can I assist you today?'}, log='Well, hello there! How can I assist you today?')

LangChain has automated the above with some additional features (e.g. error handling, logging, alos automates the above loop.)

In [99]:
from langchain.agents import AgentExecutor
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import MessagesPlaceholder
from langchain.memory import ChatMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful but sassy assistant"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# Define the function list
functions = [convert_to_openai_function(f) for f in tools]

# Create model and bind funtions
model = ChatOpenAI(temperature=0).bind(functions=functions)

chain = prompt | model | OpenAIFunctionsAgentOutputParser()

agent_chain = RunnablePassthrough.assign(
    agent_scratchpad= lambda x: format_to_openai_functions(x["intermediate_steps"])
) | chain

agent_executor = AgentExecutor(agent=agent_chain, tools=tools, verbose=True)

In [98]:
agent_executor.invoke({"input": "what is langchain?"})



> Entering new AgentExecutor chain...

Invoking: `search_wikipedia` with `{'query': 'Langchain'}`


Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Vector database
Summary: A vector database, vector store or vector search engine is a database that uses the vector space model to store vectors (fixed-length lists of numbers) along with other data items. Vector databases typically implement one or more approximate nearest neighbor algorithms, so that one can search the database with a query vector to retrieve the closest matching database records.
Vectors are mathematical representations of data in a high-dimensional space. In this space, each dimension corresponds to a feature of the

{'input': 'what is langchain?',
 'output': 'LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. It is used for document analysis and summarization, chatbots, and code analysis.'}

In [90]:
agent_executor.invoke({"input": "my name is bob"})



> Entering new AgentExecutor chain...
Nice to meet you, Bob! How can I assist you today?

> Finished chain.


{'input': 'my name is bob',
 'output': 'Nice to meet you, Bob! How can I assist you today?'}

In [91]:
agent_executor.invoke({"input": "whats my name"})



> Entering new AgentExecutor chain...
I'm sorry, I'm not able to access personal information like your name. How can I assist you today?

> Finished chain.


{'input': 'whats my name',
 'output': "I'm sorry, I'm not able to access personal information like your name. How can I assist you today?"}

### Adding Memory to the AgentExecutor

In [104]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful but sassy assistant"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [105]:
agent_chain = RunnablePassthrough.assign(
    agent_scratchpad= lambda x: format_to_openai_functions(x["intermediate_steps"])
) | prompt | model | OpenAIFunctionsAgentOutputParser()

In [106]:
from langchain.memory import ConversationBufferMemory
memory = ConversationBufferMemory(return_messages=True,memory_key="chat_history")

In [111]:
agent_executor = AgentExecutor(agent=agent_chain, 
                               tools=tools, 
                               verbose=True, 
                               memory=memory)

In [108]:
agent_executor.invoke({"input": "my name is bob"})



> Entering new AgentExecutor chain...
Nice to meet you, Bob! How can I assist you today?

> Finished chain.


{'input': 'my name is bob',
 'chat_history': [HumanMessage(content='my name is bob', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Nice to meet you, Bob! How can I assist you today?', additional_kwargs={}, response_metadata={})],
 'output': 'Nice to meet you, Bob! How can I assist you today?'}

In [109]:
agent_executor.invoke({"input": "whats my name"})



> Entering new AgentExecutor chain...
You just told me your name is Bob!

> Finished chain.


{'input': 'whats my name',
 'chat_history': [HumanMessage(content='my name is bob', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Nice to meet you, Bob! How can I assist you today?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='whats my name', additional_kwargs={}, response_metadata={}),
  AIMessage(content='You just told me your name is Bob!', additional_kwargs={}, response_metadata={})],
 'output': 'You just told me your name is Bob!'}

In [110]:
agent_executor.invoke({"input": "whats the weather in sf?"})



> Entering new AgentExecutor chain...

Invoking: `get_current_temperature` with `{'latitude': 37.7749, 'longitude': -122.4194}`


The current temperature is 14.4°CThe current temperature in San Francisco is 14.4°C.

> Finished chain.


{'input': 'whats the weather in sf?',
 'chat_history': [HumanMessage(content='my name is bob', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Nice to meet you, Bob! How can I assist you today?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='whats my name', additional_kwargs={}, response_metadata={}),
  AIMessage(content='You just told me your name is Bob!', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='whats the weather in sf?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The current temperature in San Francisco is 14.4°C.', additional_kwargs={}, response_metadata={})],
 'output': 'The current temperature in San Francisco is 14.4°C.'}

## Create a chatbot¶

the code from course is not working